# Week 6 — From Documents to Defensible Patterns

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/advanced-data-mining-course/blob/main/weeks/06/Week_06_Documents_Retrieval_Topics.ipynb)

**Driving question:** What useful structure can we find in documents before using a neural language model?

An invoice gave us explicit product codes. A document does not arrive with a uniquely correct set of items: we must decide what counts as a document, which words survive preprocessing, and how their occurrences should be weighted. Those decisions shape what we can retrieve and which themes appear. The connection to representation learning is direct: start with a transparent lexical baseline before asking what a dense embedding adds next week.

By the end, you should be able to audit a small corpus; hand-check term frequency–inverse document frequency (TF–IDF); retrieve documents using sparse vectors and cosine similarity; inspect a lexical success and limitation; and interpret non-negative matrix factorization (NMF) components through words **and actual documents**. A topic is a proposed interpretation of a weighted pattern, not a discovered ground-truth category.

Opening prediction: a search for “storing electricity for later use” should find which kind of document? Would replacing “car” with “automobile” always leave a word-based result unchanged? Write a prediction before looking at the rankings.

The provisional 180-minute core excludes breaks: opening/setup 10; Reading A 15; tiny worked example 15; Reading B 15; corpus/index demonstration 15; Investigation 1 40; Reading C 15; Investigation 2 40; synthesis 15. Reading windows include reflection. Choose one route per investigation; the other is optional. All activities are formative; this notebook introduces no assignment or deadline.

## Start with visual intuition

Use **one 3–5-minute visual starter within the opening block**, before the detailed reading. The notebook remains self-contained if an external site is unavailable.

1. **Build the representation:** [UC Berkeley D-Lab — Understanding TF–IDF](https://dlab-berkeley.github.io/demos/tfidf.html). Click **Next: See Document Collection**, then **Next: Document-Term Matrix**, then **Generate Document-Term Matrix**. Use **Toggle Full Vocabulary** to reveal more columns. **Bring back:** what do rows, columns, zeros, and repeated-word counts mean? Its three-document example shows how language becomes a sparse matrix. Weighting conventions can differ across tutorials; our worked example states the tested library's smoothing and normalization explicitly. You do not need to complete the whole tutorial before continuing.
2. **Preview the interpretation task:** [scikit-learn 1.7 — Illustrated topic extraction](https://scikit-learn.org/1.7/auto_examples/applications/plot_topics_extraction_with_nmf_lda.html). Inspect the bar charts labeled **NMF**, choose one component, and suggest a cautious label from its highest-weight words. **Bring back:** what document would you want to read before trusting your label? This is a static illustrated example, not an interactive explorer. Its other algorithm and speed comparisons are optional reference, not additional methods for today's core.

Start from a visual question, then return to the explanations and runnable examples below. Full external reading belongs outside the core budget; there is no upload or account requirement for either resource.

## Setup — a prepared snapshot, no manual corpus work

In Colab, connect to a CPU runtime and choose **Runtime → Run all**. The badge targets the student release and becomes usable after publication. Packages install automatically. If installation requests a restart, restart and run all. Locally use `uv sync --locked`, then `uv run jupyter lab` from the course root. No earlier notebook state, API key, account, GPU, ZIP upload, or page collection is required.

The data cell uses the bundled/cached Wikipedia teaching snapshot or downloads the same checksum-verified file. A compressed copy embedded in this notebook is the automatic fallback if the asset is not yet published or the connection fails. It contains exactly the same attributed snapshot, not substitute synthetic documents. Its encoded storage cell is collapsed where the notebook viewer supports it. First-time dependency installation still needs internet.

The snapshot contains 120 selected English Wikipedia introductions, with at most 500 whitespace-separated words per article. We chose linked subjects across space, energy, life, computing, cities, and economics to make retrieval and later graph investigations possible. These are collection choices, not true classes an algorithm must recover. Each record keeps its page ID, original/revision URL, Wikipedia-contributor attribution, and Creative Commons Attribution–ShareAlike 4.0 license link. See the [corpus guide](../../data/WIKIPEDIA_CORPUS.md).

Run the defaults first. For an experiment, predict the consequence, change one setting, and rerun the dependent cells. Work with the supplied loader and methods; spend the investigation on evidence and interpretation.

In [ ]:
# Keep message timestamps compatible with newer Python runtimes.
import jupyter_client.session as _jupyter_session

def _notebook_utcnow():
    from datetime import datetime, timezone
    return datetime.now(timezone.utc)

_clock_code = getattr(_jupyter_session.utcnow, '__code__', None)
if _clock_code is not None and 'utcnow' in _clock_code.co_names:
    # Update in place so existing kernel aliases receive the same fix.
    _jupyter_session.utcnow.__code__ = _notebook_utcnow.__code__

import sys
import subprocess
import importlib.util

if importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
        'numpy>=2,<3','pandas>=2.2,<3','matplotlib>=3.9,<4',
        'scikit-learn>=1.6,<1.8'])
    print('Colab dependencies ready. Continue, or restart and run all if requested.')
else:
    print('Local runtime: using the locked course environment.')

In [ ]:
from pathlib import Path
import base64
import gzip
import hashlib
import json
import urllib.request
import urllib.error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.decomposition import NMF
from sklearn.neighbors import NearestNeighbors
import sklearn

pd.set_option('display.max_colwidth', 100)
plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('NumPy:', np.__version__, '| pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)

In [ ]:
#@title Prepared corpus fallback (runs automatically; no edits needed)
# Compressed, attributed data; no executable code is encoded here.
EMBEDDED_CORPUS = '...'

In [ ]:
filename = 'wikipedia_teaching.json.gz'
DATA_URL = 'https://raw.githubusercontent.com/rrfhwn/advanced-data-mining-course/main/data/' + filename
EXPECTED_SHA256 = '8cd7e9bdcec080488f264832c15dee171729cc52df84b442eb3838ca33932de6'

def load_corpus(directory=None, url=DATA_URL):
    if directory is None:
        candidates = [folder/'data'/filename for folder in [Path.cwd(), *Path.cwd().parents]]
        path = next((p for p in candidates if p.is_file()), Path.cwd()/'data'/filename)
    else:
        path = Path(directory)/filename
    if path.is_file():
        payload, origin = path.read_bytes(), 'bundled/cache'
    else:
        try:
            with urllib.request.urlopen(url, timeout=15) as response:
                payload = response.read()
            origin = 'download'
        except (urllib.error.URLError, TimeoutError, OSError):
            payload = base64.b64decode(EMBEDDED_CORPUS)
            origin = 'embedded snapshot fallback'
        if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA256:
            raise ValueError('Snapshot checksum mismatch; do not use a changed corpus silently.')
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(payload)
    if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA256:
        raise ValueError('Cached snapshot checksum mismatch; restore the matching course snapshot.')
    document = json.loads(gzip.decompress(payload))
    table = pd.DataFrame(document['documents']).sort_values('doc_id').reset_index(drop=True)
    if table.doc_id.duplicated().any() or table.text.isna().any() or table.text.str.strip().eq('').any():
        raise ValueError('Document IDs must be unique and document text must be present.')
    print('Data source:', origin, '| Snapshot:', document['version'], '| Documents:', len(table))
    return table

documents = load_corpus()
display(documents[['doc_id', 'title', 'source_url']].head(4))

## Reading A — Words become features through choices

**Read independently for 10–15 minutes, including the checkpoint.** Your aim is to explain what a matrix entry means before treating it as evidence about meaning.

### Decide what one document represents

A corpus is a collection of documents. Here one document is the retained introduction of one selected Wikipedia article. It is not a paragraph drawn independently from a population of all writing, and it is not the entire article. A longer page can contain information absent from its introduction. Truncation can remove an important qualification or a useful search term, so a zero match tells us something about the retained text and representation, not everything the original source says.

Stable document IDs keep the text attached to its metadata when we sort, filter, retrieve, or later add embeddings and edges. A positional row number alone is fragile: sorting one table can silently attach a convincing title to another article's vector. The raw snapshot is preserved; experimental representations are separate objects. Selection-theme metadata records why we collected an article. It is neither a relevance judgment for every query nor a ground-truth topic label.

Audit the corpus before mining it. Check empty text, missing provenance, repeated IDs, exact duplicate text, and document lengths. Near duplicates need inspection rather than a rule that any high similarity proves duplication. Related articles may legitimately share definitions, and copied boilerplate can make unrelated documents look similar. Repeated documents can inflate apparent retrieval coverage or dominate a topic model. These are the text equivalents of repeated invoice lines and uncertain duplicates from earlier lectures.

### Tokenization is a representation decision

A token is a unit produced by a text-processing rule. Our word analyzer lowercases text and keeps sequences of at least two word characters. Punctuation and one-character tokens disappear under that default. A computer language written as “C++” can therefore lose its identifier; “US” and “us” become the same lowercase token. Such choices may be useful for some searches and harmful for others. They are not neutral cleaning.

A vocabulary is the set of retained feature names. A document–term matrix has one row per document and one column per vocabulary item. An entry can record a count: how many times that term occurs in that document. A zero means no retained occurrence, not that the concept is absent from the world. Most documents use only a small fraction of a collection's vocabulary, so the matrix is sparse. Sparse storage records nonzero values and their positions instead of storing every zero. It does not itself remove the conceptual limits of a word-based representation.

Stop words are words excluded by a chosen list, often because they are common function words. Removing them can reduce noise, but an English stop-word list can remove negation or a domain-relevant term. Keep the decision visible. We do not ask students to manually edit every document, build a tokenizer, or invent a universal list. The experiment compares working configurations and inspects their consequences.

### Weight frequency against a collection baseline

Term frequency tells us how often a term appears in one document. Document frequency, df(t), counts how many distinct documents contain it, regardless of repetitions within each document. Let N be the number of documents and c(t,d) the raw count of term t in document d. We use the library's smoothed inverse document frequency:

$$\mathrm{idf}(t)=1+\ln\left(\frac{1+N}{1+\mathrm{df}(t)}\right),\qquad
u(t,d)=c(t,d)\,\mathrm{idf}(t).$$

The logarithm is natural. The added ones are this implementation's smoothing convention; other tutorials may use a different formula. If a term occurs in every document, its idf is one, not zero. Repeating a term in one document raises its term frequency but does not increase its document frequency beyond that one document. A rarer term receives a larger inverse-document-frequency weight relative to a ubiquitous term in this corpus. This is a collection baseline, not proof that the rarer word is more useful for every question.

Take four constructed documents: “solar energy energy,” “wind energy,” “solar wind,” and “energy storage.” Energy occurs in three documents; solar in two; storage in one. Their idf values are about 1.223, 1.511, and 1.916. The first document's unnormalized energy weight is twice 1.223. Do not confuse this with 2 divided by the number of words: our specified term frequency is a raw count.

Finally, the default vector is divided by its Euclidean length, $\sqrt{\sum_t u(t,d)^2}$, giving a unit-length row. This L2 normalization changes the displayed weights; they no longer sum to the raw counts or necessarily sum to one. An empty retained vector cannot be normalized to unit length and stays zero. We handle that case explicitly when retrieving.

### Checkpoint

Why does “energy energy” contribute two to term count but only one document to document frequency? Which term has the largest idf in the tiny example? What information can tokenization lose? Explain why a missing term is not a judgment of irrelevance.

<details><summary>Self-check</summary>

Term frequency counts within-document repetitions; document frequency counts containing documents once. Storage is rarest and has the largest idf. Lowercasing can merge meanings, punctuation can destroy identifiers, and stop-word removal can erase negation. A term may be absent because of vocabulary selection, wording, truncation, or preprocessing.

</details>

**Optional precise reference:** [scikit-learn 1.7 — TF–IDF weighting](https://scikit-learn.org/1.7/modules/feature_extraction.html#tfidf-term-weighting). Compare its smoothed formula and normalization with the calculations below. It is documentation to consult when needed, not another required reading before the experiment.

In [ ]:
toy_texts = ['solar energy energy', 'wind energy', 'solar wind', 'energy storage']
toy_vectorizer = TfidfVectorizer(norm=None, smooth_idf=True)
toy_unnormalized = toy_vectorizer.fit_transform(toy_texts)
toy_terms = toy_vectorizer.get_feature_names_out()
toy_counts = CountVectorizer(vocabulary=toy_vectorizer.vocabulary_).transform(toy_texts)
toy_df = np.asarray((toy_counts > 0).sum(axis=0)).ravel()
hand_idf = 1 + np.log((1 + len(toy_texts)) / (1 + toy_df))
assert np.allclose(hand_idf, toy_vectorizer.idf_)
display(pd.DataFrame({'term': toy_terms, 'document_frequency': toy_df,
                       'idf': hand_idf}).round(3))
display(pd.DataFrame(toy_unnormalized.toarray(), columns=toy_terms,
                     index=[f'D{i+1}' for i in range(4)]).round(3))
toy_normalized = normalize(toy_unnormalized)
display(pd.DataFrame(toy_normalized.toarray(), columns=toy_terms).round(3))
# TODO 1: Predict the change, then add a second 'energy' to D2 and rerun this cell.
# Explain term frequency, document frequency, and row normalization separately.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
image = ax.imshow(toy_normalized.toarray(), cmap='Blues', vmin=0, vmax=1)
ax.set(xticks=np.arange(len(toy_terms)), xticklabels=toy_terms,
       yticks=np.arange(4), yticklabels=['D1', 'D2', 'D3', 'D4'],
       title='Four constructed documents: normalized TF–IDF weights')
for row in range(4):
    for col in range(len(toy_terms)):
        value = toy_normalized[row, col]
        ax.text(col, row, f'{value:.2f}', ha='center', va='center',
                color='white' if value > .6 else 'black')
fig.colorbar(image, ax=ax, label='Feature weight (not probability)')
plt.tight_layout(); plt.show()
# TODO 2: Explain one nonzero cell and one zero without calling either a topic label.

## Reading B — Retrieval is a question about useful documents

**Read and reflect for 10–15 minutes.** Explain the baseline and the evidence you would inspect before calling a search result good.

### Fix the searchable collection and its scope

Our snapshot is a deliberately bounded collection, not a representative sample of Wikipedia or general knowledge. It contains selected article introductions, capped at 500 whitespace-separated words, across six linked subject areas. Some introductions are much shorter. We retain original and revision URLs, license attribution, collection-theme metadata, and observed links to selected articles. Those links were recorded from full article pages, so a link need not occur in the retained introduction. They are saved for later graph work and are not used in today's text ranking.

We must not confuse the metadata with the model's features. The index below uses the prepared article text only. It does not append selection themes, titles, hyperlinks, or a desired answer to each document. Titles help us inspect results; IDs keep rows aligned. A title or source page may contain context missing from the extract, so use both the shown passage and the provenance when explaining a result. The source collection has an editorial style and language coverage we chose; no retrieval score establishes fairness across languages or domains absent from it.

The same fixed document IDs can later support dense embeddings and graph attributes. That continuity makes a method comparison easier to interpret, but it does not mean all those representations answer the same question. A hyperlink is an observed editorial relation; cosine similarity is a relation computed from a chosen text representation. Neither automatically measures factual agreement or causal influence.

The working index keeps words seen in at least two documents and limits the vocabulary to 4,000 features, with English stop-word removal. These boundaries keep the experiment small but can exclude a rare identifier or a useful term. We use single words first; a later route adds two-word phrases. The vocabulary and weights are fitted on the searchable documents, without using query relevance judgments. Report these choices when explaining coverage.

### Cosine compares directions in a feature space

The query is transformed with the vocabulary and inverse document frequencies fitted on the document index. We do not fit a new vocabulary on each query. For nonzero document vector x and query vector q, cosine similarity is

$$\cos(x,q)=\frac{x\cdot q}{\|x\|_2\|q\|_2}.$$

The numerator sums products of matching feature weights. If both vectors are already L2-normalized, this is their dot product. With nonnegative TF–IDF values, the score lies between zero and one up to numerical rounding. A high score means strong directional overlap under this representation. It is not a probability of relevance or a calibrated confidence level. Short lexical matches can be misleading, and longer text can contain useful information that receives less weight after normalization.

Our baseline compares raw term-count cosine with TF–IDF cosine using the same document rows and vocabulary. Count cosine already controls vector length, so the comparison specifically tests weighting by corpus prevalence, not merely whether we normalized one method. Keep the query, document texts, filtering, and number of returned results fixed. Otherwise a changed ranking may come from a changed question rather than the method you intended to test.

### A blank query vector is a coverage result

A query can lose all its features because it contains only removed words, uses unseen vocabulary, or has identifiers destroyed by tokenization. If its vector is zero, cosine has no meaningful directional interpretation. Our search helper returns an explicit status and no results. It does not present arbitrary all-zero ties as recommendations. Similarly, a document with no retained features has no lexical evidence for this query. The code reports such rows and leaves them out of positive-match rankings without erasing their IDs from the corpus.

For positive scores, ties are resolved by stable document ID so repeated runs are interpretable. A nonzero score only means at least one retained feature contributes. We inspect the terms contributing to a result and the actual text; shared words may have different meanings. One search about infections can retrieve a biological virus or a computer virus. The right choice depends on the question's context, not on the numerical score alone.

### Judge relevance separately from similarity

For each query, decide in plain language what a helpful document would need to explain. Then read several returned passages. A useful result might directly answer the question, provide background, or be only a superficial word match. Record which interpretation applies and quote only a short phrase or refer to an exact passage with its source. A selection theme is too coarse to settle this: two documents collected under “energy” can answer very different questions.

Today's supplied queries are development prompts for learning and inspection. We do not report a final accuracy or declare one representation universally superior. Next week will compare lexical and embedding retrieval on shared queries with explicit relevance judgments and a separation between development and evaluation. Trying many queries now and later claiming those same examples were untouched evaluation would repeat Week 5's selection problem.

### Checkpoint

Why does fitting a new vocabulary per query spoil the comparison? What would justify preferring the third result over the first? What should happen when the query contains no retained features? Identify a lexical failure that a semantic method might help with, and one it might not solve.

<details><summary>Self-check</summary>

The query and documents must share coordinates and corpus-derived weights. A lower-ranked result can be more helpful after reading its evidence; the score does not define relevance. A zero query vector should produce an explicit coverage message rather than arbitrary matches. Synonyms may benefit from embeddings, while negation, exact identifiers, factual reliability, and source coverage can still fail.

</details>

In [ ]:
# Corpus audit: no changes are made to the preserved text.
documents['word_count'] = documents.text.str.split().str.len()
exact_duplicates = documents.text.str.lower().str.split().str.join(' ').duplicated(keep=False)
display(documents[['word_count']].describe().round(1))
print('Repeated IDs:', int(documents.doc_id.duplicated().sum()),
      '| Empty texts:', int(documents.text.str.strip().eq('').sum()),
      '| Texts in normalized-duplicate groups:', int(exact_duplicates.sum()))
display(documents.sort_values(['word_count', 'doc_id']).head(3)[['doc_id', 'title', 'word_count', 'truncated']])

def fit_index(texts, *, stop_words='english', ngram_range=(1, 1)):
    vectorizer = TfidfVectorizer(stop_words=stop_words, ngram_range=ngram_range,
        min_df=2, max_df=1.0, max_features=4000, norm='l2', smooth_idf=True)
    matrix = vectorizer.fit_transform(texts)
    return vectorizer, matrix

vectorizer, X = fit_index(documents.text)
count_vectorizer = CountVectorizer(vocabulary=vectorizer.vocabulary_, stop_words='english')
count_X = normalize(count_vectorizer.transform(documents.text))
print('Sparse matrix:', X.shape, '| Nonzero entries:', X.nnz,
      '| Density:', round(X.nnz / (X.shape[0] * X.shape[1]), 4),
      '| Zero-vector documents:', int(np.asarray(X.getnnz(axis=1) == 0).sum()))
# TODO 3: Inspect the shortest text and the closest lexical pair before calling it a duplicate.
neighbors = NearestNeighbors(n_neighbors=2, metric='cosine', algorithm='brute').fit(X)
distances, indices = neighbors.kneighbors(X)
pair_rows, seen = [], set()
for row, matches in enumerate(indices):
    others = [(float(distances[row, j]), int(other)) for j, other in enumerate(matches) if other != row]
    if others:
        distance, other = others[0]
        pair = tuple(sorted((row, other)))
        if pair not in seen:
            seen.add(pair)
            pair_rows.append({'left': documents.iloc[pair[0]].title,
                'right': documents.iloc[pair[1]].title, 'cosine': 1-distance})
display(pd.DataFrame(pair_rows).sort_values(['cosine', 'left'], ascending=[False, True]).head(4).round(3))

## Investigation 1 — Explain why these documents were returned

**40 minutes. Choose one route.** Use the same preserved documents; loading and indexing already work. Before searching, write what a useful result should explain.

**Route A — `queries`:** compare raw-count cosine with TF–IDF cosine for the three supplied development questions. Look for a changed rank and a result whose usefulness needs closer inspection. A ranking that does not change is also a finding; explain the counts and weights instead of forcing a winner.

**Route B — `phrases`:** keep the first question fixed and compare unigram TF–IDF with a representation adding adjacent two-word features, called bigrams. The other vectorizer settings stay the same. Inspect a changed result or explain why the top results persist. A bigram is a literal word sequence, not a semantic relation.

Spend five minutes predicting, fifteen running and comparing, fifteen inspecting passages and contributing terms, and five explaining a result to someone else. **TODO 4:** select a route and record the top three IDs and your relevance judgments. **TODO 5:** inspect one result and explain its largest query-feature contributions; then run a deliberately unknown query and interpret its status. Try one query of your own only after documenting the supplied comparison.

**Done when:** you can explain two returned documents with source evidence, one limitation or disagreement between score and usefulness, and one controlled representation comparison. If results remain similar, explain what the comparison did establish and what it could not settle.

<details><summary>Hint if you are stuck</summary>

Use “The method returned this article because these retained features overlap. The passage is helpful / only background / misleading for my question because ... .” Open the original URL when you need more context, but distinguish that additional article content from what the index actually saw.

</details>

In [ ]:
DEVELOPMENT_QUERIES = [
    'storing electricity for later use',
    'how infections spread between computers',
    'travel through a city without a private car',
]

def retrieve(query, fitted, matrix, metadata, k=3):
    """Rank L2-normalized matrix rows aligned with metadata and fitted features."""
    if matrix.shape[0] != len(metadata):
        raise ValueError('Vector rows must align with document metadata.')
    if not isinstance(k, int) or isinstance(k, bool) or k < 1:
        raise ValueError('k must be a positive integer.')
    query_vector = normalize(fitted.transform([query]))
    if matrix.shape[1] != query_vector.shape[1]:
        raise ValueError('Query and document vectors must share feature coordinates.')
    if query_vector.nnz == 0:
        return pd.DataFrame(columns=['doc_id', 'title', 'score', 'snippet', 'source_url']), 'no retained query features'
    scores = np.asarray((matrix @ query_vector.T).toarray()).ravel()
    ranked = metadata[['doc_id', 'title', 'text', 'source_url']].copy()
    ranked['score'] = scores
    ranked = ranked[ranked.score > 1e-12].sort_values(['score', 'doc_id'], ascending=[False, True]).head(k)
    ranked['snippet'] = ranked.text.str.slice(0, 180)
    return ranked[['doc_id', 'title', 'score', 'snippet', 'source_url']], 'positive lexical matches' if len(ranked) else 'no positive lexical matches'

# TODO 4: Choose queries or phrases. The topic experiment still uses the baseline X.
retrieval_route = 'queries'
retrieval_results = {}
if retrieval_route == 'queries':
    for query in DEVELOPMENT_QUERIES:
        print('Question:', query)
        for name, fitted, matrix in [('count_cosine', count_vectorizer, count_X), ('tfidf_cosine', vectorizer, X)]:
            results, status = retrieve(query, fitted, matrix, documents)
            retrieval_results[(query, name)] = results
            print(name, '|', status)
            display(results[['doc_id', 'title', 'score']].round(3))
elif retrieval_route == 'phrases':
    phrase_vectorizer, phrase_X = fit_index(documents.text, ngram_range=(1, 2))
    for name, fitted, matrix in [('unigrams', vectorizer, X), ('unigrams_bigrams', phrase_vectorizer, phrase_X)]:
        results, status = retrieve(DEVELOPMENT_QUERIES[0], fitted, matrix, documents)
        retrieval_results[(DEVELOPMENT_QUERIES[0], name)] = results
        print(name, '|', status)
        display(results[['doc_id', 'title', 'score']].round(3))
else:
    raise ValueError('Choose queries or phrases.')

In [ ]:
# TODO 5: Inspect actual text and identify contributing query terms.
inspection_query = DEVELOPMENT_QUERIES[0]
inspection_results, status = retrieve(inspection_query, vectorizer, X, documents)
if len(inspection_results):
    inspect_id = inspection_results.iloc[0].doc_id
    row = int(documents.index[documents.doc_id.eq(inspect_id)][0])
    print('Inspected:', documents.iloc[row].title, '|', documents.iloc[row].source_url)
    print(documents.iloc[row].text)
    query_vector = vectorizer.transform([inspection_query])
    contribution = X[row].multiply(query_vector).tocoo()
    terms = vectorizer.get_feature_names_out()
    display(pd.DataFrame({'term': terms[contribution.col], 'dot_product_contribution': contribution.data})
            .sort_values('dot_product_contribution', ascending=False).round(4))
unknown_results, unknown_status = retrieve('qzxvnotincorpus', vectorizer, X, documents)
print('Unknown query:', unknown_status, '| Returned rows:', len(unknown_results))

# Constructed failure: bag-of-words can erase a decision-critical distinction.
negation_texts = ['renewable energy is reliable', 'renewable energy is not reliable']
negation_model = TfidfVectorizer(stop_words='english')
negation_vectors = negation_model.fit_transform(negation_texts)
print('Synthetic negation example — cosine:', float((negation_vectors[0] @ negation_vectors[1].T).toarray()[0, 0]))
print('Retained features:', list(negation_model.get_feature_names_out()))

## Reading C — A topic is a weighted pattern that needs a name and a challenge

**Read independently for 10–15 minutes, including the checkpoint.** Your goal is to interpret a topic using documents, not merely invent a label for an attractive word list.

### Factor a matrix into two useful views

Retrieval begins with a query. Topic exploration asks a different question: which combinations of terms help summarize variation across this document collection? Non-negative matrix factorization, or NMF, approximates the nonnegative document–term matrix V as a product of two nonnegative matrices:

$$V\approx WH.$$

If V has D document rows and T term columns, choosing K components gives W with shape D × K and H with shape K × T. Each row of H is a component's pattern of term weights. Each row of W describes how strongly a document uses those components in the reconstruction. Nonnegativity means components add together; one component cannot cancel another with a negative contribution. The approximation generally leaves reconstruction error. We use an established library to fit it rather than treating a tiny implementation as a reliable optimizer.

For a small constructed example, suppose one component emphasizes solar and wind, while a second emphasizes gene. A document can use both components. Its mixed weights do not mean it is a classification error or that it has secretly been assigned two real-world categories. They describe a combination in this representation. The next code cell multiplies a tiny W and H so that the dimensions and additive mechanism are visible before fitting the real corpus.

### The name is our interpretation

A component has weights; it does not arrive with an authoritative name. A list containing “energy,” “power,” “electricity,” and “storage” suggests an energy-related theme, but the label should be checked against actual documents. It might combine energy technologies, economics, or a repeated introductory phrase. Reading the strongest documents can reveal a useful common context or expose that a convenient name is too broad.

Inspect at least two components and several documents for each. Name them tentatively and say what the label captures and what it leaves out. A top document is selected because of its reconstruction weight, not because a human verified that it is the purest example of a known subject. A document with substantial contributions from multiple components can be especially informative: perhaps the collection contains a meaningful bridge between transport, energy, and economics, or perhaps common vocabulary is dominating the representation.

We collected articles across six subject areas to obtain variety. Asking for six components does not test whether the algorithm recovered six true categories. Other counts can be defensible summaries. A smaller K can combine patterns; a larger K can split a broad pattern or give a repeated term its own component. More components can improve reconstruction without yielding more useful interpretations. Do not choose K solely because a smaller error looks mathematically better or because it matches the collection tags.

### Weights are not automatically probabilities

The fitted W and H values are nonnegative but are not intrinsically calibrated probabilities. Scaling one component's H row and inversely scaling its W column can preserve their product. Topic numbers are also arbitrary: component zero in one fit need not correspond to component zero in another. Comparing raw coefficients or matching topics by their index across models can therefore produce a misleading story.

For a single fitted model, ranking words within one H row and documents within the corresponding W column is a useful inspection aid. If you normalize a document's W row to sum to one for display, those shares describe this factorization; they still are not the probability that its subject is objectively one topic. Do not use a topic weight as a replacement for a separately defined relevance judgment in retrieval.

### Challenge one choice without moving everything

Today we compare the baseline with either a different component count or a change in stop-word removal. Keep the source documents and their IDs fixed. When the vocabulary changes, the matrices have different coordinates, so raw reconstruction errors and coefficients are not directly comparable evidence of improvement. Instead, compare descriptors, representative documents, and an inspected consequence for one of your claims.

Start from a prediction. Will a broad energy pattern split when K changes? Will removing fewer common words make topic descriptions less informative, or preserve an important distinction? After fitting, look for related components by their words and documents rather than assuming equal topic numbers correspond. If the interpretation persists, say what that supports. If it changes, narrow the claim. A single seed or one parameter comparison does not establish general stability.

### Keep the conclusion within the collection

Our model summarizes a curated English-language snapshot of article introductions. It does not estimate what subjects are most important to society, what all Wikipedia contains, or what every author believes. An editorial link saved in the metadata provides another view later, not a ground-truth topic label now. The useful result is a defensible description of patterns in a specified representation, with actual examples and a limitation.

### Checkpoint

Which matrix tells you the words for a component, and which tells you its strongest documents? Why can a document use several components? Why can component zero mean different things in two runs? What would you inspect before deciding that a larger model gives better topics?

<details><summary>Self-check</summary>

H contains component–term weights; W contains document–component weights. Mixed documents use additive patterns. Component order and scale are not unique, so compare content rather than index numbers or raw coefficients. Read words and representative documents, inspect a changed consequence, and judge usefulness for the question; lower reconstruction error alone does not settle interpretation quality.

</details>

In [ ]:
# A deliberately constructed factorization: three documents, two patterns, three terms.
tiny_W = np.array([[2., 0.], [0., 3.], [1., 1.]])
tiny_H = np.array([[1., 1., 0.], [0., 0., 2.]])
display(pd.DataFrame(tiny_W @ tiny_H, columns=['solar', 'wind', 'gene'],
                     index=['D1', 'D2', 'D3']))
print('D3 combines both patterns. Neither W row sums nor H row sums are probabilities.')

def fit_topics(matrix, count=6):
    if not isinstance(count, int) or isinstance(count, bool) or not 2 <= count <= min(10, *matrix.shape):
        raise ValueError('Use 2 to 10 components, bounded by the matrix dimensions.')
    model = NMF(n_components=count, init='nndsvda', random_state=42,
                solver='mu', beta_loss='frobenius', max_iter=1200, tol=1e-4)
    weights = model.fit_transform(matrix)
    return model, weights

def topic_summary(model, weights, fitted, metadata, words=8, examples=3):
    if weights.shape[0] != len(metadata):
        raise ValueError('Topic weights and metadata must have aligned rows.')
    features = fitted.get_feature_names_out()
    rows = []
    for topic, pattern in enumerate(model.components_):
        term_indices = np.argsort(-pattern, kind='stable')[:words]
        doc_rows = np.argsort(-weights[:, topic], kind='stable')[:examples]
        rows.append({'component': topic,
            'top_words': ', '.join(features[term_indices]),
            'representative_documents': ' | '.join(metadata.iloc[doc_rows].title)})
    return pd.DataFrame(rows)

topic_model, W = fit_topics(X, 6)
summary = topic_summary(topic_model, W, vectorizer, documents)
display(summary)
print('Shapes V, W, H:', X.shape, W.shape, topic_model.components_.shape,
      '| Iterations:', topic_model.n_iter_)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
features = vectorizer.get_feature_names_out()
for topic, ax in enumerate(axes.ravel()):
    pattern = topic_model.components_[topic]
    selected = np.argsort(-pattern, kind='stable')[:7][::-1]
    ax.barh(features[selected], pattern[selected], color='#476f95')
    ax.set(title=f'Component {topic}', xlabel='Reconstruction weight (not probability)')
fig.suptitle('Patterns to inspect: read representative documents before naming components', fontsize=13)
plt.tight_layout(); plt.show()

## Investigation 2 — Name a pattern, then challenge the name

**40 minutes. Choose one route.** The baseline has six components. Inspect words and the actual strongest documents for two of them before changing anything. Write a tentative label and one document that complicates it.

**Route A — `topic_count`:** keep the same text matrix and change six components to four. Predict which broad patterns might combine. After fitting, compare words and representative documents; do not match component numbers automatically. The reconstruction objective changes with model size, so a lower error alone does not establish better interpretation.

**Route B — `stop_words`:** keep six components and the same documents, but retain common English words instead of using the supplied stop-word list. Refit the vocabulary and model. Inspect whether descriptors or representative documents change. Since the feature coordinates changed, comparing raw errors or coefficients is not a fair measure of improvement.

Spend ten minutes on baseline documents, five predicting, fifteen running/inspecting the alternative, and ten revising your interpretation. **TODO 6:** choose a route and identify a related alternative component by content. **TODO 7:** read the strongest three documents of one component and compare an explicitly chosen alternative; explain one persistence or change. The provided defaults work, but your label and conclusion must come from the displayed evidence.

**Done when:** you have two tentative component labels supported by documents, one controlled challenge, and a revised claim with a limitation. An ambiguous matching between components is worth reporting. You do not need a single objectively correct topic count or a perfect alignment to our collection themes.

<details><summary>Hint</summary>

Start with “This component emphasizes these words and these documents, so I would tentatively call it ... . After changing ..., the related pattern ..., which makes me qualify ... .” If a word list looks generic, read documents before concluding the model discovered a meaningless topic.

</details>

In [ ]:
# TODO 6: Choose topic_count or stop_words; keep source texts and IDs fixed.
topic_route = 'topic_count'
if topic_route == 'topic_count':
    alternative_vectorizer, alternative_X = vectorizer, X
    alternative_model, alternative_W = fit_topics(alternative_X, 4)
elif topic_route == 'stop_words':
    alternative_vectorizer, alternative_X = fit_index(documents.text, stop_words=None)
    alternative_model, alternative_W = fit_topics(alternative_X, 6)
else:
    raise ValueError('Choose topic_count or stop_words.')
alternative_summary = topic_summary(alternative_model, alternative_W, alternative_vectorizer, documents)
display(alternative_summary)

# TODO 7: Set an alternative component by CONTENT after reading the two summaries.
baseline_component = 0
alternative_component = 0  # This is an inspection starting point, not an asserted alignment.
for name, weights, topic in [('baseline', W, baseline_component),
                             ('alternative', alternative_W, alternative_component)]:
    if not 0 <= topic < weights.shape[1]:
        raise ValueError('Choose an existing component from the displayed summary.')
    order = np.argsort(-weights[:, topic], kind='stable')[:3]
    print(name, 'component', topic)
    for row in order:
        print(documents.iloc[row].doc_id, '|', documents.iloc[row].title,
              '| reconstruction weight:', round(float(weights[row, topic]), 3))
        print(documents.iloc[row].text[:900])
        print('Source:', documents.iloc[row].source_url)

## Synthesis — what can you defend about this collection?

Use fifteen minutes for a 150–250-word memo. **TODO 8:** state one retrieval finding and one topic interpretation, with document IDs, inspected text, a controlled comparison, and a limitation. Separate what the system calculated from your relevance judgment or topic label. A useful conclusion can say that a pattern is plausible but insufficiently specific for the proposed purpose.

Use an evidence table with `question`, `representation`, `observed_result`, `inspected_documents`, `challenge`, `revised_claim`, and `next_evidence`. Explain at least one zero-query result or lexical failure, and one reason a component name could be misleading. Revisit the opening prediction about electricity storage and synonymous wording.

A useful sentence structure is: “For this query, the representation ranked [IDs] highly because [...]. Reading the passages showed [...]. Changing [...] affected [...], so my revised claim is [...].” For topics, connect your tentative name to both weighted words and actual documents. Do not claim that six model components validate six true themes.

All classroom activities are formative. Assignment briefs and deadlines are supplied separately; this lecture does not release an assessment. Next week we keep the document IDs and compare lexical retrieval with pretrained document embeddings. The question will be what the representation adds for shared relevance questions, including exact identifiers, negation, and failures, rather than whether an embedding produces an attractive plot.

## Optional extensions — expand the evidence before expanding the model list

Try character-based word-boundary n-grams on one query involving spelling or an identifier, keeping documents and inspection criteria fixed. Explain the coverage change; do not declare a representation universally better from one selected example.

For topic inspection, choose one document that has substantial weights in two components and read it closely. Row-normalizing W can help display relative contributions within this fitted model, but those shares are not calibrated topic probabilities. A second random seed or an additional topic count is exploratory, not a required sweep.

For later graph work, inspect a saved `outgoing_doc_ids` entry and the original source page. Links refer to full articles and a selected target set, whereas our text vectors use only retained introductions. A missing selected link is not proof that two articles are unrelated. No graph pipeline or embedding download is required today.